# Extract parish features

Extract parish features used to identify comparable parishes in `05_cluster_parishes.ipynb`. Run this notebook from the project root. Outputs belong in `data/04_extract_features/`.

County population and median age use the local 2020 Census DP1 export after county lookup. Social links are counted from cached homepage external links using editable CSV rules. School affiliation is predicted from internal and external homepage links with a configurable parish limit.

| Feature | Purpose |
| --- | --- |
| `county_population` | Size of surrounding community |
| `county_median_age` | Age characteristics of community |
| `n_social_links` | Proxy for digital engagement |
| `attached_school` | Whether parish operates or is associated with a school |

The final dataset is `data/04_extract_features/7_parish_features.csv`. Section 7 also saves feature coverage and rows needing review. Run sections in order; section 6 retains the configurable school-prediction limit.


## 0 Load parish inputs

Load `data/01_build_site_list/2_parish_list.csv`, generated by `01_build_site_list.ipynb`. Preserve all source columns and use the input row index as `parish_index`, matching page discovery. Initialize the four feature columns with nullable types; missing means unavailable or not yet processed, rather than zero or false.


In [1]:
from pathlib import Path

import pandas as pd

input_path = Path("data/01_build_site_list/2_parish_list.csv")
parishes_df = pd.read_csv(input_path)
parishes_df.index.name = "parish_index"

features_df = parishes_df.reset_index().copy()
feature_dtypes = {
    "county_population": "Int64",
    "county_median_age": "Float64",
    "n_social_links": "Int64",
    "attached_school": "boolean",
}
for feature, dtype in feature_dtypes.items():
    features_df[feature] = pd.Series(pd.NA, index=features_df.index, dtype=dtype)

print(f"Loaded {len(parishes_df)} parishes from {input_path}")
features_df.head()


Loaded 864 parishes from data\01_build_site_list\2_parish_list.csv


,parish_index,name,address,site_url,diocese,category,parish_list_url,county_population,county_median_age,n_social_links,attached_school
0,0,Father Purcell Memorial Center for Exceptional...,"Montgomery, AL",https://fatherpurcell.org/,Archdiocese of Mobile,Sunday Mass Schedule,https://mobarch.org/parishfinder,<NA>,<NA>,<NA>,<NA>
1,1,Holy Spirit Catholic Parish,"Montgomery, AL",https://holyspiritmgm.org/,Archdiocese of Mobile,Sunday Mass Schedule,https://mobarch.org/parishfinder,<NA>,<NA>,<NA>,<NA>
2,2,Blessed Francis Xavier Seelos Parish,"31122 US Hwy 31, Spanish Fort, AL, 36527",https://francisxseelos.org/,Archdiocese of Mobile,Baldwin/Escambia Deanery,https://mobarch.org/parishfinder,<NA>,<NA>,<NA>,<NA>
3,3,Chapel of Our Lady of Bon Secour,"17266 County Road 49 South, Bon Secour, AL, 36511",https://www.stjohnms.com/our-lady-of-bon-secour,Archdiocese of Mobile,Baldwin/Escambia Deanery,https://mobarch.org/parishfinder,<NA>,<NA>,<NA>,<NA>
4,4,Christ the King Catholic Church,"711 College Avenue, Daphne, AL, 36526",https://ctkdaphne.org/,Archdiocese of Mobile,Baldwin/Escambia Deanery,https://mobarch.org/parishfinder,<NA>,<NA>,<NA>,<NA>


## 1 Look up parish counties

Use the [US Census Geocoder](https://geocoding.geo.census.gov/geocoder/Geocoding_Services_API.html) to resolve each distinct address to a county or county equivalent. No API key is required. This section sends parish addresses to the Census service and may take several minutes on the first run.

When Census returns `no_match`, including a cached response, retry using city/state and then the five-digit ZIP code against the [GeoNames US postal reference](https://download.geonames.org/export/zip/). Full state names and ZIP+4 are supported. Accept a fallback only when the matching reference rows identify one county; conflicting counties remain unresolved unless the ZIP lookup resolves them. GeoNames data is provided under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/); attribution: [GeoNames](https://www.geonames.org/).

Preserve unmodified Census responses and the downloaded GeoNames archive in the folder selected by `COUNTY_CACHE_DIR` in `.env` or the environment (default `data/cache/census_geocoder/`). An explicit `cache_dir` override takes precedence. Reuse them on subsequent runs. Census uses `Public_AR_Current` / `Current_Current`; cached responses retain the geography used when first downloaded. Request errors are retried on the next run.

Add `county`, the five-character `county_fips` code (including leading zeros), matched address or locality query, lookup status, lookup method, and any error to `features_df`. The new `county_lookup_method` output column distinguishes `address`, `city_state`, and `zip`. City/state and ZIP matches are approximate: postal reference coverage does not establish an exact parish location, and cities or ZIP codes can cross county boundaries. Review these matches before using them for demographic features. Multiple Census address matches do not trigger fallback.

Save parish identifiers, source addresses, and county lookup results to `data/04_extract_features/1_parish_counties.csv`. Source input files remain unchanged.


In [2]:
from utils.county_lookup import get_county_cache_dir, lookup_county

output_dir = Path("data/04_extract_features")
county_cache_dir = get_county_cache_dir()
addresses = features_df["address"].fillna("").astype(str).str.strip()
unique_addresses = addresses.drop_duplicates().tolist()
county_results = {}

for number, address in enumerate(unique_addresses, start=1):
    county_results[address] = lookup_county(address, cache_dir=county_cache_dir)
    if number % 25 == 0 or number == len(unique_addresses):
        print(f"County lookup: {number}/{len(unique_addresses)} distinct addresses")

county_columns = [
    "county", "county_fips", "county_lookup_status", "county_lookup_method",
    "county_matched_address", "county_lookup_error",
]
county_results_df = pd.DataFrame(
    [county_results[address] for address in addresses],
    index=features_df.index,
    columns=county_columns,
).astype("string")
features_df[county_columns] = county_results_df

output_dir.mkdir(parents=True, exist_ok=True)
features_df[["parish_index", "name", "diocese", "site_url", "address"] + county_columns].to_csv(
    output_dir / "1_parish_counties.csv", index=False,
)
features_df["county_lookup_status"].value_counts(dropna=False)


County lookup: 25/861 distinct addresses
County lookup: 50/861 distinct addresses
County lookup: 75/861 distinct addresses
County lookup: 100/861 distinct addresses
County lookup: 125/861 distinct addresses
County lookup: 150/861 distinct addresses
County lookup: 175/861 distinct addresses
County lookup: 200/861 distinct addresses
County lookup: 225/861 distinct addresses
County lookup: 250/861 distinct addresses
County lookup: 275/861 distinct addresses
County lookup: 300/861 distinct addresses
County lookup: 325/861 distinct addresses
County lookup: 350/861 distinct addresses
County lookup: 375/861 distinct addresses
County lookup: 400/861 distinct addresses
County lookup: 425/861 distinct addresses
County lookup: 450/861 distinct addresses
County lookup: 475/861 distinct addresses
County lookup: 500/861 distinct addresses
County lookup: 525/861 distinct addresses
County lookup: 550/861 distinct addresses
County lookup: 575/861 distinct addresses
County lookup: 600/861 distinct addre

county_lookup_status
matched      845
ambiguous     11
no_match       6
error          2
Name: count, dtype: Int64

In [3]:
# Review unresolved locations and approximate city/state or ZIP matches.
features_df.loc[
    (features_df["county_lookup_status"] != "matched")
    | (features_df["county_lookup_method"] != "address"),
    ["parish_index", "name", "address", "county", "county_fips",
     "county_lookup_status", "county_lookup_method", "county_lookup_error"],
]


,parish_index,name,address,county,county_fips,county_lookup_status,county_lookup_method,county_lookup_error
0,0,Father Purcell Memorial Center for Exceptional...,"Montgomery, AL",Montgomery,01101,matched,city_state,<NA>
1,1,Holy Spirit Catholic Parish,"Montgomery, AL",Montgomery,01101,matched,city_state,<NA>
2,2,Blessed Francis Xavier Seelos Parish,"31122 US Hwy 31, Spanish Fort, AL, 36527",Baldwin,01003,matched,city_state,<NA>
24,24,St. Martin of Tours Parish,"725 Elba Highway, Troy, AL, 36079",Pike,01109,matched,city_state,<NA>
27,27,Annunciation Parish,"Monroeville, AL",Monroe,01099,matched,city_state,<NA>
...,...,...,...,...,...,...,...,...
786,786,Christ the King,"52473 S.R. 933, South Bend, IN 46637",St. Joseph,18141,matched,city_state,<NA>
801,801,"Sacred Heart, Notre Dame","104 Sacred Heart Parish Center, Notre Dame, IN...",St. Joseph,18141,matched,city_state,<NA>
811,811,St. Catherine of Alexandria,"9989 S. STR 9, Columbia City, IN 46725",Whitley,18183,matched,city_state,<NA>
816,816,St. Gaspar del Bufalo,"10871 N. St. Rd. 9, Rome City, IN 46784",Noble,18113,matched,city_state,<NA>


## 2 county_population

Use the **2020 Census Demographic Profile (DP1)** export in `data/04_extract_features/raw/DECENNIALDP2020.DP1_2026-10-20T134923/`. The date in the folder name is the download date; the demographic reference year is **2020**.

Load `DP1_0001C` (total population) and `DP1_0073C` (median age, both sexes), as defined by the accompanying column metadata. Skip the descriptive label row and select county/county-equivalent geographies (`0500000US` plus five-digit FIPS). Preserve FIPS as strings and require one record per county before mapping to parishes.

`county_population` is a nullable integer. Missing or nonnumeric Census values remain missing. Counties that could not be located, or whose codes are absent from the 2020 export, also retain missing demographic values; no county boundary crosswalk is assumed. Approximate county assignments from section 1 remain approximate.


In [4]:
census_dir = Path("data/04_extract_features/raw/DECENNIALDP2020.DP1_2026-10-20T134923")
census_path = census_dir / "DECENNIALDP2020.DP1-Data.csv"
census_columns = {
    "GEO_ID": "geo_id",
    "NAME": "county_census_name",
    "DP1_0001C": "county_population",
    "DP1_0073C": "county_median_age",
}
county_demographics_df = pd.read_csv(census_path, usecols=list(census_columns), dtype="string",).rename(columns=census_columns)
county_demographics_df = county_demographics_df.loc[county_demographics_df["geo_id"].str.fullmatch(r"0500000US\d{5}", na=False)].copy()
county_demographics_df["county_fips"] = county_demographics_df["geo_id"].str[-5:]
county_demographics_df["county_population"] = pd.to_numeric(county_demographics_df["county_population"], errors="coerce",).astype("Int64")
county_demographics_df["county_median_age"] = pd.to_numeric(county_demographics_df["county_median_age"], errors="coerce",).astype("Float64")
county_demographics_df = county_demographics_df.set_index("county_fips", verify_integrity=True)

features_df["county_population"] = features_df["county_fips"].map(county_demographics_df["county_population"]).astype("Int64")
print(f"Loaded {len(county_demographics_df)} counties from the 2020 Census export")
print(f"Population available for {features_df['county_population'].notna().sum()}/{len(features_df)} parishes")
features_df[["parish_index", "name", "county_fips", "county_population"]].head()


Loaded 3221 counties from the 2020 Census export
Population available for 844/864 parishes


,parish_index,name,county_fips,county_population
0,0,Father Purcell Memorial Center for Exceptional...,01101,228954
1,1,Holy Spirit Catholic Parish,01101,228954
2,2,Blessed Francis Xavier Seelos Parish,01003,231767
3,3,Chapel of Our Lady of Bon Secour,01003,231767
4,4,Christ the King Catholic Church,01003,231767


## 3 county_median_age

Map the county's 2020 median age in years (`DP1_0073C`, both sexes) using the same Census table and FIPS join as population. `county_median_age` is a nullable floating-point column.

Save the current parish feature table to `data/04_extract_features/3_parish_county_features.csv`, retaining parish identifiers, county lookup provenance, the two county features, and the two remaining feature columns. Review rows missing either demographic value below. The raw Census export and section 1 county results remain unchanged.


In [5]:
features_df["county_median_age"] = features_df["county_fips"].map(
    county_demographics_df["county_median_age"]
).astype("Float64")

output_dir.mkdir(parents=True, exist_ok=True)
features_df.to_csv(output_dir / "3_parish_county_features.csv", index=False)
print(f"Median age available for {features_df['county_median_age'].notna().sum()}/{len(features_df)} parishes")
print("Saved data/04_extract_features/3_parish_county_features.csv")

# Missing county assignments and codes absent from the 2020 export need review.
features_df.loc[
    features_df[["county_population", "county_median_age"]].isna().any(axis=1),
    ["parish_index", "name", "county", "county_fips", "county_lookup_status",
     "county_population", "county_median_age"],
]


Median age available for 844/864 parishes
Saved data/04_extract_features/3_parish_county_features.csv


,parish_index,name,county,county_fips,county_lookup_status,county_population,county_median_age
36,36,Our Mother of Mercy Parish,<NA>,<NA>,ambiguous,<NA>,<NA>
99,99,St.James,<NA>,<NA>,ambiguous,<NA>,<NA>
103,103,St.Jude,<NA>,<NA>,ambiguous,<NA>,<NA>
130,130,St.Henry,<NA>,<NA>,ambiguous,<NA>,<NA>
163,163,St. John the Baptist,<NA>,<NA>,ambiguous,<NA>,<NA>
203,203,St. Philip Benizi Mission,<NA>,<NA>,no_match,<NA>,<NA>
212,212,St. Ann Parish,Skagway-Hoonah-Angoon,02232,matched,<NA>,<NA>
222,222,Holy Family Catholic Church,<NA>,<NA>,no_match,<NA>,<NA>
237,237,Sacred Heart Catholic Church,<NA>,<NA>,no_match,<NA>,<NA>
242,242,Saint Charles Spinola Catholic Church,<NA>,<NA>,no_match,<NA>,<NA>


## 4 Prepare homepage links

Read every parish homepage from the shared HTML cache selected by `HTML_CACHE_DIR` in `.env` or the environment. `CACHE_ONLY=True` uses existing files only; missing pages and parishes without a website URL are recorded for review (`cache_missing` and `missing_url`). Set it to `False` to allow the existing scraper to download missing homepages. Original and discovery-normalized cache keys are checked, and cached redirect URLs are respected. No external link destinations are downloaded.

Extract all HTTP(S) hyperlinks (`a[href]` and `area[href]`) in the saved HTML, resolve relative and protocol-relative URLs (including HTML base URLs), and remove fragments. Classify links to the parish's original and redirected hosts, `www` aliases, and their subdomains as internal; other hosts are external. Keep both types for `attached_school`, including internal school pages and school subdomains. Mailto/tel links and script-only or embedded content without hyperlinks are not included.

Deduplicate by parish and normalized destination URL, combining link labels and recording the number of occurrences as `link_count`. Paths and query strings are retained. Save `4_homepage_links.csv` with all links (`link_url`, `link_host`, `link_type`, labels, and occurrence counts). Also save `4_homepage_external_links.csv` (parish identifiers, final homepage URL, destination URL/host, link text, occurrence count) and `4_homepage_link_status.csv` (one row per parish, status, and any error). The status file retains `external_link_count` and adds `homepage_link_count` for all links. Status `ok` with zero links means a cached page was read successfully; missing or failed pages do not imply zero social engagement.


In [6]:
import requests
from dotenv import load_dotenv

from utils.external_links import HOMEPAGE_LINK_COLUMNS, LINK_COLUMNS, extract_homepage_links, read_homepage
from utils.scraping import get_cache_dir

_ = load_dotenv(".env", override=False)
CACHE_ONLY = True
html_cache_dir = get_cache_dir()
identity_columns = ["parish_index", "name", "diocese", "site_url"]
homepage_link_rows = []
homepage_status_rows = []

for number, (_, parish) in enumerate(features_df.iterrows(), start=1):
    identity = parish[identity_columns].to_dict()
    site_url = str(parish["site_url"]).strip() if pd.notna(parish["site_url"]) else ""
    status = {**identity, "homepage_url": "", "homepage_status": "ok", "homepage_link_count": pd.NA, "external_link_count": pd.NA, "error": ""}
    try:
        html, homepage_url = read_homepage(site_url, cache_dir=html_cache_dir, cache_only=CACHE_ONLY)
        links = extract_homepage_links(html, site_url, homepage_url)
        homepage_link_rows.extend({**identity, **link} for link in links)
        status.update(homepage_url=homepage_url, homepage_link_count=len(links),
                      external_link_count=sum(link["link_type"] == "external" for link in links))
    except FileNotFoundError as exc:
        status.update(homepage_status="cache_missing", error=str(exc))
    except (requests.RequestException, OSError, ValueError) as exc:
        status.update(homepage_status="error" if site_url else "missing_url", error=str(exc))
    homepage_status_rows.append(status)
    if number % 25 == 0 or number == len(features_df):
        print(f"Homepage links: {number}/{len(features_df)} parishes")

homepage_links_df = pd.DataFrame(homepage_link_rows, columns=identity_columns + HOMEPAGE_LINK_COLUMNS)
external_links_df = homepage_links_df.loc[homepage_links_df["link_type"] == "external"].rename(
    columns={"link_url": "external_url", "link_host": "external_host"},
)[identity_columns + LINK_COLUMNS].copy()
homepage_status_df = pd.DataFrame(
    homepage_status_rows,
    columns=identity_columns + ["homepage_url", "homepage_status", "homepage_link_count", "external_link_count", "error"],
)
homepage_status_df["homepage_link_count"] = homepage_status_df["homepage_link_count"].astype("Int64")
homepage_status_df["external_link_count"] = homepage_status_df["external_link_count"].astype("Int64")
output_dir.mkdir(parents=True, exist_ok=True)
homepage_links_df.to_csv(output_dir / "4_homepage_links.csv", index=False)
external_links_df.to_csv(output_dir / "4_homepage_external_links.csv", index=False)
homepage_status_df.to_csv(output_dir / "4_homepage_link_status.csv", index=False)
homepage_status_df["homepage_status"].value_counts(dropna=False)


Homepage links: 25/864 parishes
Homepage links: 50/864 parishes
Homepage links: 75/864 parishes
Homepage links: 100/864 parishes
Homepage links: 125/864 parishes
Homepage links: 150/864 parishes
Homepage links: 175/864 parishes
Homepage links: 200/864 parishes
Homepage links: 225/864 parishes
Homepage links: 250/864 parishes
Homepage links: 275/864 parishes
Homepage links: 300/864 parishes
Homepage links: 325/864 parishes
Homepage links: 350/864 parishes
Homepage links: 375/864 parishes
Homepage links: 400/864 parishes
Homepage links: 425/864 parishes
Homepage links: 450/864 parishes
Homepage links: 475/864 parishes
Homepage links: 500/864 parishes
Homepage links: 525/864 parishes
Homepage links: 550/864 parishes
Homepage links: 575/864 parishes
Homepage links: 600/864 parishes
Homepage links: 625/864 parishes
Homepage links: 650/864 parishes
Homepage links: 675/864 parishes
Homepage links: 700/864 parishes
Homepage links: 725/864 parishes
Homepage links: 750/864 parishes
Homepage link

homepage_status
ok               600
missing_url      197
cache_missing     67
Name: count, dtype: int64

## 5 n_social_links

Edit `data/04_extract_features/social_link_rules.csv` to determine which external links count. Initial rules include social media, YouTube/video services, email marketing/newsletters, Flocknote, messaging, and parish communication apps. Unmatched links remain non-social and are retained for school extraction and rule review.

Rules are processed in CSV order; first match wins. `domain` matches the exact hostname and its subdomains, with `www` treated as an alias. `domain_path_prefix` matches that domain plus a path prefix (for example, `example.com/newsletter`). Patterns omit the URL scheme. `is_social` must be `TRUE` or `FALSE`; put exclusions before broader inclusion rules. `service` and `category` are descriptive labels. Domain matching uses hostname boundaries, so lookalike domains do not match.

Reload the saved external links and apply the current rules without rereading HTML. `n_social_links` counts distinct external destination URLs classified as social per parish, including distinct pages on the same service. Repeated occurrences of a URL count once; fragments are ignored and distinct paths/queries remain distinct. Successfully read homepages with no qualifying links receive zero. Missing or failed homepages retain a missing count.

Save all annotated links to `5_classified_external_links.csv` (including rule/service/category), unmatched-domain frequencies to `5_unmatched_external_domains.csv`, and the current feature table to `5_parish_social_features.csv`. School extraction uses `4_homepage_links.csv` (or `homepage_links_df`) with both internal and external links; social counting uses only the external-link subset.


In [7]:
from utils.external_links import classify_social_links, load_social_rules

external_links_df = pd.read_csv(output_dir / "4_homepage_external_links.csv", keep_default_na=False)
homepage_status_df = pd.read_csv(output_dir / "4_homepage_link_status.csv", keep_default_na=False)
social_rules = load_social_rules(output_dir / "social_link_rules.csv")
classified_external_links_df = classify_social_links(external_links_df, social_rules)

social_counts = (
    classified_external_links_df.loc[classified_external_links_df["is_social"]]
    .groupby("parish_index")["external_url"].nunique()
)
# Only a successfully inspected homepage can establish a zero count.
successful_parishes = homepage_status_df.loc[
    homepage_status_df["homepage_status"] == "ok", "parish_index"
]
features_df["n_social_links"] = features_df["parish_index"].map(social_counts).fillna(0).astype("Int64")
features_df.loc[~features_df["parish_index"].isin(successful_parishes), "n_social_links"] = pd.NA

unmatched_domains_df = (
    classified_external_links_df.loc[classified_external_links_df["matched_social_rule"] == ""]
    .groupby("external_host", as_index=False)
    .agg(n_parishes=("parish_index", "nunique"), n_links=("external_url", "size"))
    .sort_values(["n_parishes", "external_host"], ascending=[False, True])
)
classified_external_links_df.to_csv(output_dir / "5_classified_external_links.csv", index=False)
unmatched_domains_df.to_csv(output_dir / "5_unmatched_external_domains.csv", index=False)
features_df.to_csv(output_dir / "5_parish_social_features.csv", index=False)
print(f"Social-link counts available for {features_df['n_social_links'].notna().sum()}/{len(features_df)} parishes")
features_df[["parish_index", "name", "n_social_links"]].head(20)


Social-link counts available for 600/864 parishes


,parish_index,name,n_social_links
0,0,Father Purcell Memorial Center for Exceptional...,0
1,1,Holy Spirit Catholic Parish,2
2,2,Blessed Francis Xavier Seelos Parish,2
3,3,Chapel of Our Lady of Bon Secour,<NA>
4,4,Christ the King Catholic Church,4
5,5,Our Lady of the Gulf Catholic Church,6
6,6,Shrine of the Holy Cross & St. John Mission,4
7,7,St Robert Bellarmine Catholic Church,2
8,8,St. Agatha Parish,1
9,9,St. Bartholomew,1


## 6 attached_school

Predict whether each parish operates or is associated with a preschool, elementary, middle, or high school. Reuse **all internal and external homepage links** from `4_homepage_links.csv`, including non-social links. Internal school pages and school subdomains are evidence alongside links to separate school websites. Rerun section 4 to generate this file before running school prediction. Each LLM prompt contains parish identity plus every URL, its path, link type, and link text. Targets are not downloaded and links are not truncated. The prompt distinguishes a parish school from general school directories, unrelated schools, and religious education.

Set `PARISH_LIMIT = 10` to inspect the first ten parish input rows; use another nonnegative integer or `None` for all rows. Missing or failed homepages are skipped and remain missing. A successfully read homepage with no HTTP(S) links still receives a prediction from an empty link list. Unselected parishes and model errors also remain missing rather than false.

Use the provider, model, and cache configured in `.env` through `utils.llm.LLMClient`. The response is validated as `yes` or `no`, then mapped to nullable boolean `attached_school`. This is a prediction from links, not verified school affiliation; no supporting school links produces a predicted no. Calls use temperature zero and a versioned prompt. Identical inputs reuse the shared LLM cache; changed links, model, or prompt version create a new request. Cached predictions count toward the row limit, so repeated limited runs inspect the same parishes. Increase the limit to include more rows.

Save yes/no results, nullable booleans, prediction statuses/errors, and provider/model/version to `6_school_predictions.csv`. Save the updated feature table to `6_parish_school_features.csv`, retaining all earlier features and county provenance. School extractor version `2.0.0` invalidates the previous external-only LLM cache. Existing cached responses are retained; this expanded prompt generates new predictions. Social-link classification still uses only the external-link file.


In [8]:
from utils.llm import LLMClient
from utils.school_prediction import extract_school_features

PARISH_LIMIT = None  # First N parish rows; None processes all rows.
llm = LLMClient()  # Provider, model, and cache settings come from .env.

school_homepage_links_df = pd.read_csv(
    output_dir / "4_homepage_links.csv", keep_default_na=False,
)
school_homepage_status_df = pd.read_csv(
    output_dir / "4_homepage_link_status.csv", keep_default_na=False,
)
school_predictions_df = extract_school_features(
    features_df,
    school_homepage_links_df,
    school_homepage_status_df,
    llm=llm,
    parish_limit=PARISH_LIMIT,
)
features_df["attached_school"] = features_df["parish_index"].map(
    school_predictions_df.set_index("parish_index")["attached_school"]
).astype("boolean")

school_predictions_df.to_csv(output_dir / "6_school_predictions.csv", index=False)
features_df.to_csv(output_dir / "6_parish_school_features.csv", index=False)
school_predictions_df.loc[
    school_predictions_df["school_prediction_status"] != "not_processed",
    ["parish_index", "name", "school_prediction", "attached_school",
     "school_prediction_status", "school_prediction_error"],
]


School 1: Father Purcell Memorial Center for Exceptional Children: no
School 2: Holy Spirit Catholic Parish: yes
School 3: Blessed Francis Xavier Seelos Parish: no
School 5: Christ the King Catholic Church: yes
School 6: Our Lady of the Gulf Catholic Church: yes
School 7: Shrine of the Holy Cross & St. John Mission: no
School 8: St Robert Bellarmine Catholic Church: no
School 9: St. Agatha Parish: no
School 10: St. Bartholomew: yes
School 11: St. John the Baptist Parish: no
School 12: St. Joseph Catholic Church: yes
School 13: St. Lawrence Parish: no
School 14: St. Margaret of Scotland Parish: no
School 16: St. Patrick Parish: yes
School 17: St. Thomas by the Sea Catholic Church: no
School 20: Our Lady of Guadalupe Mission: no
School 21: St. Columba Catholic Church: no
School 22: St. John Parish: no
School 23: St. John the Evangelist Parish: no
School 24: St. Joseph Parish: no
School 25: St. Martin of Tours Parish: no
School 28: Annunciation Parish: no
School 29: Cathedral-Basilica of 

,parish_index,name,school_prediction,attached_school,school_prediction_status,school_prediction_error
0,0,Father Purcell Memorial Center for Exceptional...,no,False,ok,
1,1,Holy Spirit Catholic Parish,yes,True,ok,
2,2,Blessed Francis Xavier Seelos Parish,no,False,ok,
3,3,Chapel of Our Lady of Bon Secour,,<NA>,homepage_unavailable,
4,4,Christ the King Catholic Church,yes,True,ok,
...,...,...,...,...,...,...
859,859,St. Therese,yes,True,ok,
860,860,"St. Thérèse, Little Flower",no,False,ok,
861,861,St. Thomas the Apostle,no,False,ok,
862,862,"St. Vincent de Paul, Elkhart",yes,True,ok,


## 7 Review and save features

Validate one row per input parish in the original order, nullable feature types, and nonnegative population/social-link counts and plausible median ages (0?120 years). Keep missing values without imputation. Limited school runs may leave predictions missing; they do not prevent exporting the other features.

The final dataset contains the original parish identifiers/source columns, county lookup provenance, and these four feature columns:

| Column | Type | Definition |
| --- | --- | --- |
| `county_population` | nullable integer | 2020 Census DP1 total county population |
| `county_median_age` | nullable float | 2020 Census DP1 county median age in years, both sexes |
| `n_social_links` | nullable integer | Distinct homepage external URLs matching the social-link rules |
| `attached_school` | nullable boolean | LLM yes/no prediction of parish school affiliation from internal and external links |

Also include homepage status and school prediction status, error, provider, model, and extractor version so downstream analysis can distinguish unavailable data from negative predictions. County demographics have reference year 2020; school and social features describe the cached homepage information used for this run. CSV missing values are blank. Read `county_fips` with `dtype="string"` to preserve leading zeros and apply `feature_dtypes` when reloading feature columns.

Outputs in `data/04_extract_features/`:

- `7_parish_features.csv`: all parish rows with four features and provenance; input for clustering.
- `7_feature_coverage.csv`: available/missing counts and percent available for each feature.
- `7_feature_review.csv`: rows missing any feature or using an approximate city/state or ZIP county assignment. `missing_features` names unavailable columns; review these rows before clustering.

Raw source files, homepage caches, external-link evidence, and model response caches remain unchanged by this export. Review missing-data coverage and approximate county assignments before selecting a clustering dataset.


In [9]:
feature_columns = list(feature_dtypes)
assert features_df["parish_index"].notna().all(), "Missing parish identifiers"
assert features_df["parish_index"].is_unique, "Duplicate parish identifiers"
assert features_df["parish_index"].tolist() == parishes_df.index.tolist(), "Parish rows or order changed"

for feature, dtype in feature_dtypes.items():
    features_df[feature] = features_df[feature].astype(dtype)
for feature in ["county_population", "n_social_links"]:
    assert features_df[feature].dropna().ge(0).all(), f"Negative values in {feature}"
assert features_df["county_median_age"].dropna().between(0, 120).all(), "Invalid median age"

# Preserve evidence availability and prediction provenance alongside the features.
homepage_metadata = homepage_status_df.set_index("parish_index", verify_integrity=True)
school_metadata = school_predictions_df.set_index("parish_index", verify_integrity=True)
for column in ["homepage_status"]:
    features_df[column] = features_df["parish_index"].map(homepage_metadata[column])
for column in [
    "school_prediction_status", "school_prediction_error", "school_llm_provider",
    "school_llm_model", "school_extractor_version",
]:
    features_df[column] = features_df["parish_index"].map(school_metadata[column])

feature_coverage_df = pd.DataFrame({
    "feature": feature_columns,
    "available": [features_df[column].notna().sum() for column in feature_columns],
    "missing": [features_df[column].isna().sum() for column in feature_columns],
})
feature_coverage_df["percent_available"] = (
    feature_coverage_df["available"].div(len(features_df) or 1).mul(100).round(1)
)
missing_features = features_df[feature_columns].isna().apply(
    lambda row: ", ".join(row.index[row]), axis=1,
)
approximate_county = features_df["county_lookup_method"].isin(["city_state", "zip"])
feature_review_df = features_df.loc[missing_features.ne("") | approximate_county].copy()
feature_review_df["missing_features"] = missing_features.loc[feature_review_df.index]

output_dir.mkdir(parents=True, exist_ok=True)
features_df.to_csv(output_dir / "7_parish_features.csv", index=False)
feature_coverage_df.to_csv(output_dir / "7_feature_coverage.csv", index=False)
feature_review_df.to_csv(output_dir / "7_feature_review.csv", index=False)
print(f"Saved {len(features_df)} parishes to {output_dir / '7_parish_features.csv'}")
print(f"{features_df[feature_columns].notna().all(axis=1).sum()} parishes have all four features")
print(f"{len(feature_review_df)} parish rows need review for missing features or approximate counties")
feature_coverage_df


Saved 864 parishes to data\04_extract_features\7_parish_features.csv
589 parishes have all four features
345 parish rows need review for missing features or approximate counties


,feature,available,missing,percent_available
0,county_population,844,20,97.7
1,county_median_age,844,20,97.7
2,n_social_links,600,264,69.4
3,attached_school,600,264,69.4
